# CNN 卷积神经网络（下）

In [12]:
import torch
from torchvision import transforms
from torchvision import datasets
from torch.utils.data import DataLoader
import torch.nn.functional as F
import torch.optim as optim

In [13]:
batch_size  = 64
transform = transforms.Compose([ 
    transforms.ToTensor(), 
    transforms.Normalize((0.1307,), (0.3081,)),
])

train_dataset = datasets.MNIST(
    '../mnist', 
    train=True, 
    download=True, 
    transform=transform) 
train_loader = DataLoader(
    train_dataset, 
    batch_size=batch_size, 
    shuffle=True)

test_dataset = datasets.MNIST(
    '../mnist', 
    train=False, 
    download=True, 
    transform=transform)
test_loader = DataLoader(
    test_dataset, 
    batch_size=batch_size, 
    shuffle=True)

## inception module
先构建一个小网络作为最终网络中频繁使用的部分
以下示例实现inception中的四条分支，四条通路分别为平均池化、1*1卷积，5*5卷积和3*3卷积，各自分支的变量名为branch_pool 、branch1x1、 branch5x5、 branch3x3

<img src="img/inception.png" width="30%">

In [16]:
class InceptionA(torch.nn.Module):
    def __init__(self,in_channels): # 添加in_channels作为实例化后可选择的参数
        # 指定进入卷积层的通道数
        super(InceptionA, self).__init__()
        self.branch_pool = torch.nn.Conv2d(in_channels,24,kernel_size = 1)

        self.branch1x1 = torch.nn.Conv2d(in_channels,16,kernel_size=1)

        self.branch5x5_1 = torch.nn.Conv2d(in_channels,16,kernel_size=1)
        self.branch5x5_2 = torch.nn.Conv2d(16,24,kernel_size=5,padding=2)

        self.branch3x3_1 = torch.nn.Conv2d(in_channels,16,kernel_size=1)
        self.branch3x3_2 = torch.nn.Conv2d(16,24,kernel_size=3,padding=1)
        self.branch3x3_3 = torch.nn.Conv2d(24,24,kernel_size=3,padding=1)

    def forward(self, x):
        branch_pool = F.avg_pool2d(x,kernel_size = 3,stride = 1,padding = 1)
        # 平均池化计算池化核中的均值，设置padding=1不改变尺寸大小
        branch_pool = self.branch_pool(branch_pool)

        branch1x1 = self.branch1x1(x)

        branch5x5 = self.branch5x5_1(x)
        branch5x5 = self.branch5x5_2(branch5x5)

        branch3x3 = self.branch3x3_1(x)
        branch3x3 = self.branch3x3_2(branch3x3)
        branch3x3 = self.branch3x3_3(branch3x3)

        outputs = [branch_pool,branch1x1,branch5x5,branch3x3]
        return torch.cat(outputs,dim=1) # 将列表的张量按索引1（即channel）拼接

In [18]:
class Net(torch.nn.Module):
    def __init__(self):
        super(Net, self).__init__()
        self.conv1 = torch.nn.Conv2d(1,10,kernel_size=5)
        self.conv2 = torch.nn.Conv2d(88,20,kernel_size=5)

        self.incep1 = InceptionA(in_channels=10)
        self.incep2 = InceptionA(in_channels=20)

        self.mp = torch.nn.MaxPool2d(2)
        self.fc = torch.nn.Linear(1408,10) 
        # 1408=88*4*4,四个分支拼合张量88通道，尺寸4*4

    def forward(self, x):
            in_size = x.size(0) # MNIST数据集输入为28*28的图片
            x = F.relu(self.mp(self.conv1(x))) # 第一个普通卷积层得到10通道
            # 图像边长28-5+1=24，再池化，12
            x = self.incep1(x) # 第一个Inception模块得到88通道
            x = F.relu(self.mp(self.conv2(x))) # 第二个普通卷积层将88通道变为20通道
            # 图像边长12-5+1=8，再池化，4
            x = self.incep2(x) # 第二个Inception模块得到88通道
            x = x.view(in_size, -1) # 88*4*4=1408 对应线性层参数
            x = self.fc(x)
            return x

model = Net()
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
model.to(device)

Net(
  (conv1): Conv2d(1, 10, kernel_size=(5, 5), stride=(1, 1))
  (conv2): Conv2d(88, 20, kernel_size=(5, 5), stride=(1, 1))
  (incep1): InceptionA(
    (branch_pool): Conv2d(10, 24, kernel_size=(1, 1), stride=(1, 1))
    (branch1x1): Conv2d(10, 16, kernel_size=(1, 1), stride=(1, 1))
    (branch5x5_1): Conv2d(10, 16, kernel_size=(1, 1), stride=(1, 1))
    (branch5x5_2): Conv2d(16, 24, kernel_size=(5, 5), stride=(1, 1), padding=(2, 2))
    (branch3x3_1): Conv2d(10, 16, kernel_size=(1, 1), stride=(1, 1))
    (branch3x3_2): Conv2d(16, 24, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (branch3x3_3): Conv2d(24, 24, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  )
  (incep2): InceptionA(
    (branch_pool): Conv2d(20, 24, kernel_size=(1, 1), stride=(1, 1))
    (branch1x1): Conv2d(20, 16, kernel_size=(1, 1), stride=(1, 1))
    (branch5x5_1): Conv2d(20, 16, kernel_size=(1, 1), stride=(1, 1))
    (branch5x5_2): Conv2d(16, 24, kernel_size=(5, 5), stride=(1, 1), padding=(2, 2))
   

In [19]:
criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.01, momentum=0.5)

In [20]:
def train(epoch):
    running_loss = 0.0
    for batch_idx, data in enumerate(train_loader, 0):
        inputs, labels = data
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        if batch_idx % 300 == 299:
            print('[%d, %5d] loss: %.3f' % (epoch + 1, batch_idx + 1, running_loss / 300))
            running_loss = 0.0

In [21]:
def test():
    correct = 0
    total = 0
    with torch.no_grad():
        for data in test_loader:
            images, labels = data
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs.data, dim=1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
        print("测试集的准确率：{}".format(correct / total))

In [22]:
%%time
if __name__=="__main__":
    for epoch in range(10):
        train(epoch)
        test()

[1,   300] loss: 0.924
[1,   600] loss: 0.217
[1,   900] loss: 0.134
测试集的准确率：0.9643
[2,   300] loss: 0.108
[2,   600] loss: 0.093
[2,   900] loss: 0.088
测试集的准确率：0.9788
[3,   300] loss: 0.081
[3,   600] loss: 0.068
[3,   900] loss: 0.070
测试集的准确率：0.9836
[4,   300] loss: 0.061
[4,   600] loss: 0.059
[4,   900] loss: 0.064
测试集的准确率：0.9858
[5,   300] loss: 0.050
[5,   600] loss: 0.056
[5,   900] loss: 0.055
测试集的准确率：0.9872
[6,   300] loss: 0.047
[6,   600] loss: 0.051
[6,   900] loss: 0.046
测试集的准确率：0.9865
[7,   300] loss: 0.044
[7,   600] loss: 0.043
[7,   900] loss: 0.043
测试集的准确率：0.987
[8,   300] loss: 0.041
[8,   600] loss: 0.039
[8,   900] loss: 0.042
测试集的准确率：0.9871
[9,   300] loss: 0.036
[9,   600] loss: 0.036
[9,   900] loss: 0.039
测试集的准确率：0.9879
[10,   300] loss: 0.033
[10,   600] loss: 0.037
[10,   900] loss: 0.033
测试集的准确率：0.9885
CPU times: total: 1min 39s
Wall time: 1min 46s


## ResNet

In [ ]:
class ResidualBlock(torch.nn.Module):
    def __init__(self,channels):
        super(ResidualBlock, self).__init__()
        self.channels = channels
        self.conv1 = torch.nn.Conv2d(channels, channels, kernel_size=3, padding=1)
        self.conv2 = torch.nn.Conv2d(channels, channels, kernel_size=3, padding=1)

    def forward(self, x):
        y = F.relu(self.conv1(x))
        y = self.conv2(y)
        return F.relu(x+y) # 先求和再激活
    # 相加要求x和y张量在各个维度尺寸都相同

In [ ]:
class Net(torch.nn.Module):
    def __init__(self):
        super(Net, self).__init__()
        self.conv1 = torch.nn.Conv2d(1,16,kernel_size=5)
        self.conv2 = torch.nn.Conv2d(16,32,kernel_size=5)
        self.mp = torch.nn.MaxPool2d(2)

        self.rblock1 = ResidualBlock(16)
        self.rblock2 = ResidualBlock(32)

        self.fc = torch.nn.Linear(512,10)

    def forward(self, x):
        in_size = x.size(0)
        x = self.mp(F.relu(self.conv1(x)))
        x = self.rblock1(x)
        x = self.mp(F.relu(self.conv2(x)))
        x = self.rblock2(x)
        x = x.view(in_size,-1)
        x = self.fc(x)
        return x

